# The reproduction gate, in miniature

This notebook regenerates **every** figure of the demo from the f3dasm record
`data/` and nothing else. No pickles, no `.npy` files, no numbers typed by hand.

If a result cannot be redrawn from here, it is not a result — it is an anecdote.


In [1]:
import matplotlib
matplotlib.use("Agg")   # scripted execution: save figures, never show them

import matplotlib.pyplot as plt
import numpy as np
from f3dasm import ExperimentData

from make_data import X_HIGH, X_LOW, true_mean, true_sd

data = ExperimentData.from_file("data")
input_df, output_df = data.to_pandas()
x = input_df["x"].to_numpy(float)
y = output_df["y"].to_numpy(float)
grid = np.linspace(X_LOW, X_HIGH, 400)

print(f"record data/: {len(data)} rows")
print("output columns:", list(output_df.columns))

record data/: 60 rows
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_hblr', 'sd_hblr', '_source_hblr', 'y_pred_selected', 'sd_selected', 'd_mean_selected', 'd_noise_selected', 'link_selected', '_source_selected']


## Baseline (written by `baseline.py`)

Plotted from the stored columns `y_pred_baseline` and `sd_baseline` — the
notebook does not re-fit anything here.

In [2]:
order = np.argsort(x)
mu_b = output_df["y_pred_baseline"].to_numpy(float)[order]
sd_b = float(output_df["sd_baseline"].iloc[0])

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(x, y, s=18, color="#333333", zorder=3, label="training data")
ax.plot(x[order], mu_b, color="#1f77b4", lw=2, label="baseline mean (deg 2)")
ax.fill_between(x[order], mu_b - 2 * sd_b, mu_b + 2 * sd_b, color="#1f77b4",
                alpha=0.20, label=f"baseline $\\pm2$ sd (constant, sd = {sd_b:.1f})")
ax.plot(grid, true_mean(grid) + 2 * true_sd(grid), "k--", lw=1.4,
        label="true $\\pm2$ sd (sd = 0.5 x)")
ax.plot(grid, true_mean(grid) - 2 * true_sd(grid), "k--", lw=1.4)
ax.set_xlabel("speed x [m/s]"); ax.set_ylabel("stopping distance y [m]")
ax.set_title("Baseline: right mean, wrong noise")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout(); fig.savefig("figures/baseline.png", dpi=150)
print("redrew figures/baseline.png from the record")

redrew figures/baseline.png from the record


---

## Agent sections below

Whoever changes the model or selects its hyperparameters appends cells here,
**between this marker and the final cell**. Those cells may only read columns
that are already in the record — `y_pred_hblr`, `sd_hblr`, `y_pred_selected`,
`sd_selected`, … — plus `data/` itself. They must not read `data_test/`, and
they must not load anything from outside the record.

<!-- AGENT CELLS BELOW THIS LINE -->

## The model that replaced the constant band (`blocks/heteroscedastic.py`)

Same degree-2 mean as the baseline — only the noise changed, from one number to
a fitted function `sd(x) = exp(polynomial in log x)`. The predictive sd also
carries the epistemic term from the Gaussian posterior over the mean weights.

These are the **hand-picked** hyperparameters, kept here as the module defaults
so the next section has something to argue with. The section after this one
picks them from the data instead.

The cell re-fits from `data/` (allowed: the fit reads the record and nothing
else) and then checks the re-fit against the stored `y_pred_hblr` / `sd_hblr`
columns. If those two ever disagree, the figure and the record have drifted
apart and the number on the slide is an anecdote.

In [3]:
from blocks.heteroscedastic import fit as fit_hblr, plot_model

model = fit_hblr(x, y, verbose=True)      # module defaults, re-fitted from `data/`

# reproduction gate: does the re-fit reproduce what the record stores?
err_mu = np.abs(model["mu_of"](x) - output_df["y_pred_hblr"].to_numpy(float)).max()
err_sd = np.abs(model["sd_of"](x) - output_df["sd_hblr"].to_numpy(float)).max()
print(f"\nre-fit vs stored columns: max |dmu| = {err_mu:.2e}, max |dsd| = {err_sd:.2e}")
assert max(err_mu, err_sd) < 1e-6, "the figure and the record have drifted apart"

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_model(ax, x, y, model, sd_b)
ax.set_title(f"Heteroscedastic BLR: the band fans with the data "
             f"(d_mean={model['d_mean']}, d_noise={model['d_noise']}, "
             f"link='{model['link']}')")
fig.tight_layout(); fig.savefig("figures/model.png", dpi=150)
print("redrew figures/model.png from the record")

HETEROSCEDASTIC BLR  d_mean=2, d_noise=1, link='log', alpha=1e-06
  noise weights c          = [2.1142 1.7022]   (converged = True)
  posterior mean weights w = [246.9621 403.4289 161.275 ]
  log evidence             = -276.3516   (5 parameters)
  train RMSE               = 26.8653
  fitted sd(x =    3)       =   1.9741   (truth 1.5000)
  fitted sd(x =   43)       =  23.3860   (truth 21.5000)
  fitted sd(x =   83)       =  47.1454   (truth 41.5000)
  epistemic share of var   = 4.330%  (mean over the training x)

re-fit vs stored columns: max |dmu| = 1.14e-13, max |dsd| = 7.11e-15
redrew figures/model.png from the record


---

## Selecting the hyperparameters (`blocks/selection.py`)

The degrees above were chosen by a person with an opinion. Here they are chosen
by a rule, over 20 candidates: `d_mean` ∈ 1..4, `d_noise` ∈ 0..2, and whether
the noise polynomial is in `log x` or in `x`.

The criterion is **6-fold cross-validated log predictive density on `data/`** —
not RMSE (every candidate has a polynomial mean, so RMSE barely moves and is
blind to the noise model) and not the training log evidence (the noise weights
are point-optimised inside it, so it under-pays for a flexible noise model; it
is plotted on the right for exactly that contrast). The winner is then taken by
the **one-standard-error rule**: among every candidate within 1 s.e. of the best
score, keep the one with the fewest parameters.

`d_noise = 0` is a constant sd — the baseline's noise model — so the baseline
is a *candidate in this table*, not an outsider to it.

The study is re-run here from `data/` alone (~3 s, 20 candidates × 7 fits) and
checked against the stored `study_selection/` record.

In [4]:
from blocks.selection import choose, plot_selection, run_study

study = run_study(x, y)                   # re-run from `data/` alone
si, so = study.to_pandas()
table = si.join(so)
verdict = choose(table)
best, pick = verdict["best"], verdict["pick"]

print("STUDY_SELECTION, regenerated here (sorted by the criterion, cv_lpd)")
print(table.sort_values("cv_lpd", ascending=False)
           .to_string(float_format=lambda v: f"{v:9.4f}"))
print(f"\nbest        : d_mean={int(best['d_mean'])}, d_noise={int(best['d_noise'])},"
      f" link='{best['link']}'  cv_lpd = {best['cv_lpd']:.4f} +/- {best['cv_lpd_se']:.4f}")
print(f"1-s.e. floor: {verdict['threshold']:.4f}  "
      f"({len(verdict['within'])} of {len(table)} candidates qualify)")
print(f"SELECTED    : d_mean={int(pick['d_mean'])}, d_noise={int(pick['d_noise'])},"
      f" link='{pick['link']}'  ({int(pick['n_params'])} parameters)")

fig = plot_selection(table, verdict)
fig.savefig("figures/selection.png", dpi=150)
print("\nredrew figures/selection.png from the record")

STUDY_SELECTION, regenerated here (sorted by the criterion, cv_lpd)
    d_mean  d_noise link    cv_lpd  cv_lpd_se   cv_rmse  log_evidence  train_rmse  n_params _source_selection
6        2        1  log   -4.4306     0.1537   29.5746     -276.3516     26.8653    5.0000         selection
8        2        2  log   -4.4322     0.1529   29.5912     -276.2374     26.8657    6.0000         selection
7        2        1  lin   -4.4428     0.1391   29.5011     -277.3907     26.8681    5.0000         selection
13       3        2  log   -4.4442     0.1525   30.3420     -280.5317     26.8723    7.0000         selection
11       3        1  log   -4.4458     0.1552   30.3084     -280.7152     26.8664    6.0000         selection
9        2        2  lin   -4.4553     0.1514   29.6189     -276.6668     26.8657    6.0000         selection
12       3        1  lin   -4.4591     0.1407   30.5827     -281.4955     26.8629    6.0000         selection
18       4        2  log   -4.4627     0.1473   30.4


redrew figures/selection.png from the record


In [5]:
import os

# Gate 1: does the re-run agree with the stored study record?
if os.path.isdir("study_selection"):
    ssi, sso = ExperimentData.from_file("study_selection").to_pandas()
    stored = ssi.join(sso).sort_values(["d_mean", "d_noise", "link"])
    here = table.sort_values(["d_mean", "d_noise", "link"])
    drift = max(float(np.abs(stored[c].to_numpy(float)
                             - here[c].to_numpy(float)).max())
                for c in ("cv_lpd", "cv_lpd_se", "log_evidence", "train_rmse"))
    print(f"re-run vs stored study_selection/: max drift = {drift:.2e} "
          f"over {len(stored)} candidates")
    assert drift < 1e-8, "the study is not reproducible -- check the fold seed"

# Gate 2: does `data/` name the same model this notebook just selected?
dm, dn = int(output_df["d_mean_selected"].iloc[0]), int(output_df["d_noise_selected"].iloc[0])
lk = str(output_df["link_selected"].iloc[0])
print(f"data/ says selected = (d_mean={dm}, d_noise={dn}, link='{lk}'); "
      f"this notebook selected = ({int(pick['d_mean'])}, {int(pick['d_noise'])}, "
      f"'{pick['link']}')")
assert (dm, dn, lk) == (int(pick["d_mean"]), int(pick["d_noise"]), str(pick["link"]))

selected = fit_hblr(x, y, dm, dn, lk)
err = max(np.abs(selected["mu_of"](x) - output_df["y_pred_selected"].to_numpy(float)).max(),
          np.abs(selected["sd_of"](x) - output_df["sd_selected"].to_numpy(float)).max())
print(f"re-fit vs stored y_pred_selected / sd_selected: max error = {err:.2e}")
assert err < 1e-6

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_model(ax, x, y, selected, sd_b, label="selected")
ax.set_title(f"Selected by 6-fold CV + 1-s.e. rule: "
             f"d_mean={dm}, d_noise={dn}, link='{lk}'")
fig.tight_layout(); fig.savefig("figures/selected.png", dpi=150)
print("redrew figures/selected.png from the record")

re-run vs stored study_selection/: max drift = 3.55e-15 over 20 candidates
data/ says selected = (d_mean=2, d_noise=1, link='log'); this notebook selected = (2, 1, 'log')
re-fit vs stored y_pred_selected / sd_selected: max error = 1.14e-13
redrew figures/selected.png from the record


---

## What the record remembers

In [6]:
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 40)

print(f"data/  --  {len(data)} rows, {len(output_df.columns)} output columns")
print("input columns :", list(input_df.columns))
print("output columns:", list(output_df.columns))

print("\nfirst 5 rows of the record (every number below was written by a tool):")
print(input_df.join(output_df).head().to_string(float_format=lambda v: f"{v:10.4f}"))

stamps = [c for c in output_df.columns if c.startswith("_source")]
print("\nprovenance -- who wrote what:")
for c in stamps:
    cols = [k for k in output_df.columns
            if k.endswith("_" + c.removeprefix("_source_")) or k == c]
    print(f"  {c:22s} {output_df[c].iloc[0]:10s} -> {cols}")

print("\nthe decision the record remembers:")
print(f"  d_mean_selected  = {int(output_df['d_mean_selected'].iloc[0])}")
print(f"  d_noise_selected = {int(output_df['d_noise_selected'].iloc[0])}")
print(f"  link_selected    = '{output_df['link_selected'].iloc[0]}'")
print(f"  chosen from {len(table)} candidates in study_selection/ by "
      f"6-fold CV + the 1-s.e. rule")

print("\nother records in this folder:")
for name in ("data_test", "study_selection"):
    if os.path.isdir(name):
        d_i, d_o = ExperimentData.from_file(name).to_pandas()
        print(f"  {name:16s} {len(d_i):4d} rows, outputs: {list(d_o.columns)}")
    else:
        print(f"  {name:16s} does not exist")

data/  --  60 rows, 13 output columns
input columns : ['x']
output columns: ['y', 'y_pred_baseline', 'sd_baseline', '_source_baseline', 'y_pred_hblr', 'sd_hblr', '_source_hblr', 'y_pred_selected', 'sd_selected', 'd_mean_selected', 'd_noise_selected', 'link_selected', '_source_selected']

first 5 rows of the record (every number below was written by a tool):
           x          y  y_pred_baseline  sd_baseline _source_baseline  y_pred_hblr    sd_hblr _source_hblr  y_pred_selected  sd_selected  d_mean_selected  d_noise_selected link_selected _source_selected
0     3.0000     5.2314           4.7319      27.5632         baseline       4.8082     1.9741         hblr           4.8082       1.9741           2.0000            1.0000           log         selected
1    43.0000   277.2732         246.9755      27.5632         baseline     246.9621    23.3860         hblr         246.9621      23.3860           2.0000            1.0000           log         selected
2    63.0000   462.5856     